# Notebook 3 — Explore Real Network Log Support

## Goal

Understand Zeek connection logs and investigate how our app could
analyze them.

## What we will do

1. Explore a sample Zeek connection record.
2. Check which measurements are available and what they mean.
3. Find compatible labeled data for training and evaluation.
4. Train and evaluate a model using those measurements.
5. Add log support to the app if the results justify it.

## Important limitation

Our existing model expects UNSW-NB15 features. It cannot directly
analyze Zeek logs.

Similar column names do not guarantee identical measurements.
Supporting the log format alone does not prove that a model can
reliably detect attacks in real traffic.

In [2]:
# Explore selected fields from a Zeek documentation example
import pandas as pd

example = pd.DataFrame([{
    "duration": 0.254115,
    "proto": "tcp",
    "orig_pkts": 6,
    "resp_pkts": 4,
    "orig_bytes": 77,
    "resp_bytes": 295
}])

display(example)

,duration,proto,orig_pkts,resp_pkts,orig_bytes,resp_bytes
0,0.254115,tcp,6,4,77,295


## Read sample Zeek JSON log entries

In [3]:
# Read two shortened connection records from Zeek's documentation
from io import StringIO
import pandas as pd

sample_log = """
{"proto":"udp","duration":0.066852,"orig_pkts":2,"resp_pkts":2,"orig_bytes":62,"resp_bytes":141}
{"proto":"tcp","duration":0.254115,"orig_pkts":6,"resp_pkts":4,"orig_bytes":77,"resp_bytes":295}
""".strip()

zeek_connections = pd.read_json(
    StringIO(sample_log),
    lines=True
)

print("Connections read:", len(zeek_connections))
display(zeek_connections)

Connections read: 2


,proto,duration,orig_pkts,resp_pkts,orig_bytes,resp_bytes
0,udp,0.066852,2,2,62,141
1,tcp,0.254115,6,4,77,295


## Inspect the labeled Zeek log

In [4]:
# Show the header and first connection records
from pathlib import Path

log_path = Path(
    "C:/Projects/21_Cyber_Threat_Detection/data/zeek/benign_4_1.log"
)

with log_path.open(encoding="utf-8") as log_file:
    for line_number, line in enumerate(log_file):
        if line_number >= 12:
            break
        print(line.rstrip())

#separator \x09
#set_separator	,
#empty_field	(empty)
#unset_field	-
#path	conn
#open	2019-12-05-15-46-36
#fields	ts	uid	id.orig_h	id.orig_p	id.resp_h	id.resp_p	proto	service	duration	orig_bytes	resp_bytes	conn_state	local_orig	local_resp	missed_bytes	history	orig_pkts	orig_ip_bytes	resp_pkts	resp_ip_bytes	tunnel_parents   label   detailed-label
#types	time	string	addr	port	addr	port	enum	string	interval	count	count	string	bool	bool	count	string	count	count	count	count	set[string]   string   string
1540469302.538640	CGm6jB4dXK71ZDWUDh	192.168.1.132	58687	216.239.35.4	123	udp	-	0.114184	48	48	SF	-	-	0	Dd	1	76	1	76	-   benign   -
1540469197.400159	CnaDAG3n5r8eiG4su2	192.168.1.132	1900	239.255.255.250	1900	udp	-	160.367579	7536	0	S0	-	-	0	D	24	8208	0	0	-   benign   -
1540469385.734089	CUrxU238nt0m6yTgKf	192.168.1.132	32893	216.239.35.8	123	udp	-	0.016986	48	48	SF	-	-	0	Dd	1	76	1	76	-   benign   -
1540469831.302625	CGQf8t1kjdxB5PHXL4	192.168.1.132	53395	2.16.60.82	443	tcp	-	0.003497	0	0	SF

# Read the labeled log into a table

In [5]:
# Read column names from the header and skip metadata lines
import pandas as pd

with log_path.open(encoding="utf-8") as log_file:
    for line in log_file:
        if line.startswith("#fields"):
            column_names = line.split()[1:]
            break

zeek_df = pd.read_csv(
    log_path,
    sep=r"\s+",
    comment="#",
    names=column_names,
    na_values=["-", "(empty)"]
)

print("Rows and columns:", zeek_df.shape)
display(zeek_df.head(3))
print("Labels:")
print(zeek_df["label"].value_counts(dropna=False))

Rows and columns: (452, 23)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed-label
0,1.540469e+09,CGm6jB4dXK71ZDWUDh,192.168.1.132,58687,216.239.35.4,123,udp,NaN,0.114184,48.0,...,NaN,0,Dd,1,76,1,76,NaN,benign,NaN
1,1.540469e+09,CnaDAG3n5r8eiG4su2,192.168.1.132,1900,239.255.255.250,1900,udp,NaN,160.367579,7536.0,...,NaN,0,D,24,8208,0,0,NaN,benign,NaN
2,1.540469e+09,CUrxU238nt0m6yTgKf,192.168.1.132,32893,216.239.35.8,123,udp,NaN,0.016986,48.0,...,NaN,0,Dd,1,76,1,76,NaN,benign,NaN


Labels:
label
benign    452
Name: count, dtype: int64


## Check connection counts and labels


In [6]:
# Count all connections and their known labels
print("Rows and columns:", zeek_df.shape)
print("\nLabel counts:")
print(zeek_df["label"].value_counts(dropna=False))

Rows and columns: (452, 23)

Label counts:
label
benign    452
Name: count, dtype: int64


## Read the second capture and check its labels

In [7]:
# Read the second log using its own header
capture_path = log_path.parent / "capture_20_1.log"

with capture_path.open(encoding="utf-8") as log_file:
    for line in log_file:
        if line.startswith("#fields"):
            capture_columns = line.split()[1:]
            break

capture_df = pd.read_csv(
    capture_path,
    sep=r"\s+",
    comment="#",
    names=capture_columns,
    na_values=["-", "(empty)"]
)

print("Rows and columns:", capture_df.shape)
print("\nLabel counts:")
print(capture_df["label"].value_counts(dropna=False))

Rows and columns: (3209, 23)

Label counts:
label
Benign       3193
Malicious      16
Name: count, dtype: int64


## Check labels in capture 34-1

In [8]:
# Read capture 34-1 using its own column header
capture_34_path = log_path.parent / "capture_34_1.log"

with capture_34_path.open(encoding="utf-8") as log_file:
    for line in log_file:
        if line.startswith("#fields"):
            columns_34 = line.split()[1:]
            break

capture_34_df = pd.read_csv(
    capture_34_path,
    sep=r"\s+",
    comment="#",
    names=columns_34,
    na_values=["-", "(empty)"]
)

print("Rows and columns:", capture_34_df.shape)
print("\nLabel counts:")
print(capture_34_df["label"].value_counts(dropna=False))

Rows and columns: (23145, 23)

Label counts:
label
Malicious    21222
Benign        1923
Name: count, dtype: int64


## Combine captures and standardize labels

In [9]:
# Keep the capture identity so we can split data responsibly later
zeek_data = pd.concat(
    [
        zeek_df.assign(capture_id="benign_4_1"),
        capture_df.assign(capture_id="capture_20_1"),
        capture_34_df.assign(capture_id="capture_34_1")
    ],
    ignore_index=True
)

# Make "benign" and "Benign" consistent
zeek_data["label"] = zeek_data["label"].str.strip().str.lower()

display(pd.crosstab(
    zeek_data["capture_id"],
    zeek_data["label"]
))
print("Total connections:", len(zeek_data))

label,benign,malicious
capture_id,,
benign_4_1,452,0
capture_20_1,3193,16
capture_34_1,1923,21222


Total connections: 26806


## Choose fearures and seperate answers

In [10]:
# Use measurements that Zeek connection logs provide
numeric_features = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts",
    "orig_ip_bytes",
    "resp_ip_bytes"
]

categorical_features = ["proto", "service", "conn_state"]

zeek_X = zeek_data[numeric_features + categorical_features].copy()
zeek_y = zeek_data["label"].map({"benign": 0, "malicious": 1})

print("Clues:", zeek_X.shape)
print("Answers:", zeek_y.shape)
print("\nMissing values:")
print(zeek_X.isna().sum())

Clues: (26806, 10)
Answers: (26806,)

Missing values:
duration         18710
orig_bytes       18710
resp_bytes       18710
orig_pkts            0
resp_pkts            0
orig_ip_bytes        0
resp_ip_bytes        0
proto                0
service          24120
conn_state           0
dtype: int64


## Check where values are missing

In [11]:
# Percentage of missing values among normal and malicious connections
missing_by_label = (
    zeek_X.isna()
    .groupby(zeek_data["label"])
    .mean()
    .mul(100)
    .round(1)
)

display(
    missing_by_label[
        ["duration", "orig_bytes", "resp_bytes", "service"]
    ]
)

,duration,orig_bytes,resp_bytes,service
label,,,,
benign,32.3,32.3,32.3,81.2
malicious,79.6,79.6,79.6,92.3


# Define the preprocessing

In [12]:
# Define preprocessing; we will fit it after splitting the captures
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_preparation = SimpleImputer(
    strategy="median",
    add_indicator=True
)

text_preparation = Pipeline([
    ("fill_missing", SimpleImputer(
        strategy="constant",
        fill_value="unknown"
    )),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

zeek_preprocessor = ColumnTransformer([
    ("numeric", numeric_preparation, numeric_features),
    ("text", text_preparation, categorical_features)
])

print("Preprocessing defined. Nothing fitted yet.")

Preprocessing defined. Nothing fitted yet.


# Separate training and evaluation captures


In [13]:
# Keep all connections from capture 20-1 out of training
evaluation_mask = zeek_data["capture_id"].eq("capture_20_1")

zeek_X_train = zeek_X.loc[~evaluation_mask].copy()
zeek_y_train = zeek_y.loc[~evaluation_mask].copy()

zeek_X_eval = zeek_X.loc[evaluation_mask].copy()
zeek_y_eval = zeek_y.loc[evaluation_mask].copy()

print("Training connections:", len(zeek_X_train))
print(zeek_y_train.value_counts().sort_index())

print("\nEvaluation connections:", len(zeek_X_eval))
print(zeek_y_eval.value_counts().sort_index())

Training connections: 23597
label
0     2375
1    21222
Name: count, dtype: int64

Evaluation connections: 3209
label
0    3193
1      16
Name: count, dtype: int64


# Train a Zeek connection detector

In [14]:
# Fit preprocessing and Random Forest using training data only
from sklearn.ensemble import RandomForestClassifier

zeek_detector = Pipeline([
    ("preprocessing", zeek_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

zeek_detector.fit(zeek_X_train, zeek_y_train)

print("Zeek detector trained.")

Zeek detector trained.


# Evaluate a seperate capture

In [15]:
# Count correct predictions and mistakes on capture 20-1
from sklearn.metrics import confusion_matrix

zeek_eval_predictions = zeek_detector.predict(zeek_X_eval)

tn, fp, fn, tp = confusion_matrix(
    zeek_y_eval,
    zeek_eval_predictions,
    labels=[0, 1]
).ravel()

print("Normal connections correctly left alone:", tn)
print("False alarms:", fp)
print("Missed malicious connections:", fn)
print("Malicious connections caught:", tp)
print("False-alarm rate:", f"{100 * fp / (tn + fp):.1f}%")

Normal connections correctly left alone: 3192
False alarms: 1
Missed malicious connections: 2
Malicious connections caught: 14
False-alarm rate: 0.0%


# Inspect the three mistakes

In [16]:
# Examine the false alarm and the two missed malicious connections
mistake_mask = zeek_eval_predictions != zeek_y_eval.to_numpy()

mistakes = zeek_data.loc[
    zeek_X_eval.index[mistake_mask],
    numeric_features + categorical_features + ["label", "detailed-label"]
].copy()

mistakes["prediction"] = zeek_eval_predictions[mistake_mask]

display(mistakes)

,duration,orig_bytes,resp_bytes,orig_pkts,resp_pkts,orig_ip_bytes,resp_ip_bytes,proto,service,conn_state,label,detailed-label,prediction
557,NaN,NaN,NaN,0,1,0,88,tcp,NaN,OTH,benign,NaN,1
2277,41420.482745,18320.0,18320.0,8295,8699,348808,366587,tcp,NaN,SF,malicious,C&C-Torii,0
3659,44325.615625,19431.0,19431.0,8872,9307,372893,388975,tcp,NaN,S1,malicious,C&C-Torii,0


## inspect another unseen capture

In [17]:
# Read capture 8-1 using its own header
capture_8_path = log_path.parent / "capture_8_1.log"

with capture_8_path.open(encoding="utf-8") as log_file:
    for line in log_file:
        if line.startswith("#fields"):
            columns_8 = line.split()[1:]
            break

capture_8_df = pd.read_csv(
    capture_8_path,
    sep=r"\s+",
    comment="#",
    names=columns_8,
    na_values=["-", "(empty)"]
)

capture_8_df["label"] = (
    capture_8_df["label"].str.strip().str.lower()
)

print("Rows and columns:", capture_8_df.shape)
print("\nLabel counts:")
print(capture_8_df["label"].value_counts(dropna=False))

Rows and columns: (10403, 23)

Label counts:
label
malicious    8222
benign       2181
Name: count, dtype: int64


## Evaluate the unchanged model on capture 8-1

In [18]:
# Evaluate without retraining or changing the threshold
capture_8_X = capture_8_df[
    numeric_features + categorical_features
].copy()

capture_8_y = capture_8_df["label"].map({
    "benign": 0,
    "malicious": 1
})

capture_8_predictions = zeek_detector.predict(capture_8_X)

tn, fp, fn, tp = confusion_matrix(
    capture_8_y,
    capture_8_predictions,
    labels=[0, 1]
).ravel()

print("Normal connections correctly left alone:", tn)
print("False alarms:", fp)
print("Missed malicious connections:", fn)
print("Malicious connections caught:", tp)
print("False-alarm rate:", f"{100 * fp / (tn + fp):.2f}%")
print("Attack recall:", f"{100 * tp / (tp + fn):.2f}%")

Normal connections correctly left alone: 2179
False alarms: 2
Missed malicious connections: 4
Malicious connections caught: 8218
False-alarm rate: 0.09%
Attack recall: 99.95%


## Inspect mistakes in capture 8-1

In [19]:
# Inspect the two false alarms and four missed malicious connections
mistake_mask_8 = (
    capture_8_predictions != capture_8_y.to_numpy()
)

mistakes_8 = capture_8_df.loc[
    mistake_mask_8,
    numeric_features + categorical_features
    + ["label", "detailed-label"]
].copy()

mistakes_8["prediction"] = capture_8_predictions[mistake_mask_8]

display(mistakes_8)

,duration,orig_bytes,resp_bytes,orig_pkts,resp_pkts,orig_ip_bytes,resp_ip_bytes,proto,service,conn_state,label,detailed-label,prediction
11,NaN,NaN,NaN,0,1,0,88,tcp,NaN,OTH,benign,NaN,1
202,NaN,NaN,NaN,0,1,0,88,tcp,NaN,OTH,benign,NaN,1
2497,3.158703,0.0,0.0,3,0,180,0,tcp,NaN,S0,malicious,C&C,0
3289,3.118724,0.0,0.0,3,0,180,0,tcp,NaN,S0,malicious,C&C,0
7808,3.118724,0.0,0.0,3,0,180,0,tcp,NaN,S0,malicious,C&C,0
8203,3.158700,0.0,0.0,3,0,180,0,tcp,NaN,S0,malicious,C&C,0


## Check detection by attack type

In [21]:
# Measure detection separately for each malicious activity in capture 8-1
attack_review = capture_8_df.loc[
    capture_8_y.eq(1), ["detailed-label"]
].copy()

attack_review["caught"] = capture_8_predictions[
    capture_8_y.eq(1).to_numpy()
]

attack_summary = attack_review.groupby("detailed-label")["caught"].agg(
    connections="size",
    caught="sum"
)

attack_summary["missed"] = (
    attack_summary["connections"] - attack_summary["caught"]
)
attack_summary["recall_percent"] = (
    100 * attack_summary["caught"] / attack_summary["connections"]
).round(2)

display(attack_summary)

,connections,caught,missed,recall_percent
detailed-label,,,,
C&C,8222,8218,4,99.95


## Evaluation findings

The model was trained on captures benign_4_1 and capture_34_1.

It was evaluated without retraining on two separate captures:

| Capture | Normal connections | False alarms | Malicious connections | Missed |
|---|---:|---:|---:|---:|
| 20-1 | 3,193 | 1 | 16 | 2 |
| 8-1 | 2,181 | 2 | 8,222 | 4 |

Capture 8-1 achieved 99.95% recall for its C&C-labeled connections.

Limitations:
- Only two separate evaluation captures were tested.
- Connections within a capture may be highly similar.
- These results do not establish reliability on everyday networks
  or across all attack types.

## Save the evaluated Zeek Pipeline

In [22]:
# Save the unchanged model and the input columns it expects
import joblib

zeek_model_path = Path(
    "C:/Projects/21_Cyber_Threat_Detection/models/zeek_detector.joblib"
)
zeek_model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "pipeline": zeek_detector,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "warning_threshold": 0.5,
        "training_captures": ["benign_4_1", "capture_34_1"]
    },
    zeek_model_path
)

print("Saved to:", zeek_model_path)

Saved to: C:\Projects\21_Cyber_Threat_Detection\models\zeek_detector.joblib


## Verify the saved ZEek detector

In [23]:
# Reload the saved pipeline and compare predictions
zeek_saved = joblib.load(zeek_model_path)

reloaded_predictions = zeek_saved["pipeline"].predict(capture_8_X)

assert (
    reloaded_predictions == capture_8_predictions
).all(), "Predictions changed after saving."

print("Passed: all 10,403 predictions match.")

Passed: all 10,403 predictions match.
